# Libraries

In [ ]:
import pandas as pd
import numpy as np
import datetime as dt
import logging
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
import talib as ta

from torch import optim
from torch.utils.data import DataLoader, Dataset, TensorDataset, random_split
from tqdm import tqdm

from numpy.lib.stride_tricks import sliding_window_view
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from torch.optim import AdamW
from utils import scale, inverse_scale, inspect
from utils.paths import CHECKPOINTS_DIR
from pypfopt import risk_models, expected_returns, plotting, EfficientFrontier

In [ ]:
from config import *
from entities import *
from strategies import *
from datasets import *
from engine import Engine
from models import DiffusionTransformer, Diffusion

# Setup

In [ ]:
logging.basicConfig(level=logging.DEBUG)
logging.getLogger('matplotlib').setLevel(logging.WARNING)

In [ ]:
cfg = TrainConfig(epochs=1000, window_size=64)
window_size = cfg.window_size
device = cfg.device
batch_size = cfg.batch_size
epochs = cfg.epochs
sim_steps = cfg.steps_to_sim
num_sims = cfg.num_sims

# Optimizer
weight_decay = cfg.optimizer.weight_decay
lr = cfg.optimizer.lr

time_range = {
    'start_date': '2021-01-01',
    'end_date': '2024-12-31'
}

ddpm = {
    'timesteps': int(1000),
    'beta_start': 0.0001,
    'beta_end': 0.02
}

ddpm_transformer = {
    'n_features': 1,
    'n_cond': 10,
    'window_size': window_size,
    'd_model': 64,
    'nhead': 4,
    'num_layers': 256,
    'dim_feedforward': 512,
    'dropout': 0.1
}

# Data [N, W, A, F]
**[N, T, A, F]** means: 
* **N**: Num of Window or Num of Batch
* **W**: Window
* **A**: Assets
* **F**: Features or Channels

In [ ]:
def time_range_info(df):
    info = (df.index.min(), df.index.max())
    print(f"Data range: {info[0]} to {info[1]}")
    
    duration = df.index.max() - df.index.min()
    print(f"Total duration: {duration}")

def time_range_mask(df, start_date, end_date):
    mask = (df.index >= start_date) & (df.index <= end_date)
    return mask

In [ ]:
symbols = ['AAPL', 'TSLA', 'MSFT', 'NVDA', 'GOOGL', 'AMZN', 'GOOG', 'META', 'AVGO', 'ORCL', 'CRM', 'ADBE', 'AMD', 'CSCO']
freq = "1d"

# Basket
basket = Basket(symbols=symbols)
basket.load_all_assets(freq=freq)

print(f"Basket data shape: {basket.data.shape}")
basket.data.head(5)

### Time Range Custom

In [ ]:
time_range_info(basket.data)

for symbol, asset in basket.assets.items():
    mask = time_range_mask(asset.data, time_range['start_date'], time_range['end_date'])
    asset.data = asset.data[mask]

time_range_info(basket.data)

## Features/Channels ($F$)
1. Find Joint Distribution $F_{\text{date\ A}} \cap F_{\text{date\ B}}$ with intersection
2. Select $F$ to norm as Return values

In [ ]:
targets = ["Close"]
features = basket.get_unique_features()
print(f"Features:\t{features}\nTargets:\t{targets}")

In [ ]:
print(f"Basket data shape before Joint: {basket.data.shape}")

joint_strategy = IntersectionStrategy()
basket.align(joint_strategy)

print(f"Basket data shape after Joint: {basket.data.shape}")

In [ ]:
basket.to_returns(features=targets, log=True, keep=False)
targets = basket.get_keyword_features("Returns")
features = basket.get_unique_features()

print(f"Features:\t{features}\nTargets:\t{targets}")
basket.data.head(5)

### Add Indicators as Features ($F$)

In [ ]:
# Indicator
time_prd = 20
fast_prd, slow_prd, signal_prd = 12, 26, 9

for symbol, asset in basket.assets.items():
    df = asset.data 
    
    for target in targets:
        s = df[target]
        
        df[f"SMA_{time_prd} {target}"] = ta.SMA(s, timeperiod=time_prd)
        df[f"EMA_{time_prd} {target}"] = ta.EMA(s, timeperiod=time_prd)
        df[f"RSI_{time_prd} {target}"] = ta.RSI(s, timeperiod=time_prd)
        
        macd, signal, hist = ta.MACD(s, fastperiod=fast_prd, slowperiod=slow_prd, signalperiod=signal_prd)
        df[f"MACD {target}"] = macd
        df[f"MACD_Sig {target}"] = signal
        df[f"MACD_Hist {target}"] = hist

print(f"Basket data shape: {basket.data.shape}")
basket.data.head(5)

In [ ]:
basket.align(joint_strategy)

print(f"Basket data shape: {basket.data.shape}")
basket.data.head(5)

### Filter only Target Features ($F_{target} $)

In [ ]:
targets = basket.get_keyword_features("Returns")
print(f"Targets: {targets}")


for symbol, asset in basket.assets.items():
    mask = asset.data.columns.isin(targets)
    asset.data = asset.data.loc[:, mask]

print(f"Basket shape: {basket.data.shape}")
basket.data.head(5)

## Dataset & Dataloader

In [ ]:
n_obs = len(basket.data)
n_assets = basket.data.columns.levels[0].size
n_features = basket.data.columns.levels[1].size

print(n_obs, n_assets, n_features)

basket_np = basket.data.values.reshape(n_obs, n_assets, n_features)
basket_np.shape

### Ratio Dataset

In [ ]:
ratios = [0.8, 0.1, 0.1]
total_count = len(basket_np)
train_count = int(total_count * ratios[0])
val_count = int(total_count * ratios[1])
test_count = total_count - train_count - val_count

print(f"Ratios DS\nTrain:\t{train_count}\nVal:\t{val_count}\nTest:\t{test_count}\nTotal:\t{total_count}")

In [ ]:
end_val = train_count + val_count

# Ratios
train_part = basket_np[:train_count]
val_part = basket_np[train_count:end_val]
test_part = basket_np[end_val:]

print(f"Train: {train_part.shape}\nVal: {val_part.shape}\nTest:{test_part.shape}")

### Scale Dataset

In [ ]:
def scale(part: np.ndarray, scaler) -> np.ndarray:
    T, A, F = part.shape
    part_2d = part.reshape(-1, F)

    # print(f"2D Part: {part_2d.shape}")

    scaled_part = scaler.transform(part_2d).reshape(T, A, F)
    return scaled_part.astype(np.float32)

def inverse_scale(scaled_part: np.ndarray, scaler) -> np.ndarray:
    original_shape = scaled_part.shape
    F = original_shape[-1]

    part_2d = scaled_part.reshape(-1, F)
    
    unscaled_2d = scaler.inverse_transform(part_2d)
    return unscaled_2d.reshape(original_shape).astype(np.float32)

def inverse_scale_with_cond(x, x_cond, scaler):
    if torch.is_tensor(x):
        x = x.cpu().numpy()
    if torch.is_tensor(x_cond):
        x_cond = x_cond.cpu().numpy()
        
    assert x.shape[:-1] == x_cond.shape[:-1], f"Shape Mismatch: x {x.shape} vs cond {x_cond.shape}"
    
    # [..., 1] + [..., 6] -> [..., 7]
    full_features = np.concatenate([x, x_cond], axis=-1)

    original_shape = full_features.shape
    total_features = original_shape[-1]

    flat_data = full_features.reshape(-1, total_features)

    unscaled_flat = scaler.inverse_transform(flat_data)

    unscaled_full = unscaled_flat.reshape(original_shape)
    unscaled_price = unscaled_full[..., 0:1]
    return unscaled_price.astype(np.float32)

In [ ]:
def inspect_data(data, name):
    if isinstance(data, torch.Tensor):
        data = data.detach().cpu().numpy()

    _min = np.min(data)
    _max = np.max(data)
    _mean = np.mean(data)
    _std = np.std(data)
    
    print(f"--- Inspecting: {name} ---")
    print("-" * 36)
    print(f"Shape: {data.shape}")
    print(f"Min:   {_min:.4f}")
    print(f"Max:   {_max:.4f}")
    print(f"Mean:  {_mean:.4f}")
    print(f"Std:   {_std:.4f}")
    print("-" * 36)
    return _min, _max, _mean, _std

In [ ]:
# scaler = MinMaxScaler(feature_range=(-1, 1))
scaler = StandardScaler()

# Require 2D Numpy Array
T, A, F = train_part.shape
scaler.fit(train_part.reshape(-1, F))

scaled_train_part = scale(train_part, scaler)
scaled_val_part = scale(val_part, scaler)
scaled_test_part = scale(test_part, scaler)

inspect_data(scaled_train_part, "Scaled Train Part")
inspect_data(scaled_val_part, "Scaled Val Part")
inspect_data(scaled_test_part, "Scaled Test Part")
print(f"Train:\t{scaled_train_part.shape}\nVal:\t{scaled_val_part.shape}\nTest:\t{scaled_test_part.shape}")

### Dataloader

In [ ]:
train_ds = MarketDataset(scaled_train_part, window_size=window_size)
val_ds = MarketDataset(scaled_val_part, window_size=window_size)
test_ds = MarketDataset(scaled_test_part, window_size=window_size)

print(f"Num of Windows\nTrain DS: {len(train_ds)}, Val Ds: {len(val_ds)}, Test DS: {len(test_ds)}\n")
print(f"A sample shape from Train DS\n\tx: {train_ds[0]['x'].shape},\n\tx_cond: {train_ds[0]['x_cond'].shape}")

In [ ]:
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False)

batch = next(iter(train_loader))
print(len(train_loader))
print(batch["x"].shape)
print(batch["x_cond"].shape)

# Model, Engine
Use *Condition DDPM* 

In [ ]:
# n_window mean batch size
n_window, window, n_assets, n_features = batch["x"].shape
n_window, window, n_assets, n_conds = batch["x_cond"].shape
ddpm_transformer['n_cond'] = n_conds

print(n_assets, n_features, n_conds)

input_channels = n_assets * n_features
cond_channels = n_assets * n_conds
print(input_channels, cond_channels)

In [ ]:
model = DiffusionTransformer(
    n_features=input_channels,
    n_cond=cond_channels,        
    window_size=window_size,             
    d_model=ddpm_transformer['d_model'],                
    nhead=ddpm_transformer['nhead'],
    num_layers=ddpm_transformer['num_layers'],
    dim_feedforward=ddpm_transformer['dim_feedforward'],
    dropout=ddpm_transformer['dropout']
).to(device)

In [ ]:
diffusion = Diffusion(model, timesteps=ddpm['timesteps'], beta_start=ddpm['beta_start'], beta_end=ddpm['beta_end']).to(device)

In [ ]:
optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

In [ ]:
engine = Engine(
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    model=diffusion,
    scaler=scaler,
    optimizer=optimizer,
    device=device,
    file_name=f"ddpm_transformer_d{ddpm_transformer['d_model']}_l{ddpm_transformer['num_layers']}",
)

In [ ]:
engine.fit(epochs)

In [ ]:
batch = next(iter(test_loader))
x_real_raw = batch["x"].to(device).float()      # [32, 64, 14, 1]
cond_raw = batch["x_cond"].to(device).float()   # [32, 64, 14, 6]
B, W, A, F = x_real_raw.shape

# Mask (Logic: 1=known, 0=predict)
mask = torch.ones_like(x_real_raw)
mask[:, -10:, :, :] = 0  # latest 10 Assets 

# Reshape Flat
# x_real: [32, 64, 14, 1] -> [32, 64, 14]
x_start_flat = x_real_raw.view(B, W, -1)

# cond: [32, 64, 14, 6] -> [32, 64, 84]
cond_flat = cond_raw.view(B, W, -1)

# mask: [32, 64, 14, 1] -> [32, 64, 14]
mask_flat = mask.view(B, W, -1)

print(f"x_start_flat: {x_start_flat.shape}\ncond_flat: {cond_flat.shape}\nmask_flat: {mask_flat.shape}")

# Inpaint
diffusion.eval()
with torch.no_grad():
    inpainted_flat = diffusion.sample_inpaint(
        x_cond=cond_flat,
        x_start=x_start_flat,
        mask=mask_flat
    )

# Reshape
# [32, 64, 14] -> [32, 64, 14, 1]
inpainted_final = inpainted_flat.view(B, W, A, F)

print("Inpaint Finished!")

In [ ]:
inpainted_final.shape

In [ ]:
def forward_simulate(steps: int, batch, diffusion: Diffusion):
    device = next(diffusion.parameters()).device
    
    x_real_raw = batch["x"].to(device).float()       # [B, W, A, F]
    cond_raw = batch["x_cond"].to(device).float()    # [B, W, A, Cond_F]

    B, W, A, F = x_real_raw.shape

    # Mask (Logic: 1=known, 0=predict)
    mask = torch.ones_like(x_real_raw)
    if steps > 0:
        mask[:, -steps:, :, :] = 0

    # Flatten
    # [B, W, A, F] -> [B, W, A*F]
    x_start_flat = x_real_raw.view(B, W, -1)
    cond_flat = cond_raw.view(B, W, -1)
    mask_flat = mask.view(B, W, -1)

    # Inpaint Execution
    diffusion.eval()
    with torch.no_grad():
        inpainted_flat = diffusion.sample_inpaint(
            x_cond=cond_flat,
            x_start=x_start_flat,
            mask=mask_flat
        )

    # Unflatten
    # [B, W, A*F] -> [B, W, A, F]
    inpainted_final = inpainted_flat.view(B, W, A, F)

    # Extract Only Predicted Part
    if steps > 0:
        # [B, steps, A, F] (32, steps, 14, 1)
        simulated_part = inpainted_final[:, -steps:, :, :]
    else:
        simulated_part = torch.empty((B, 0, A, F), device=device)

    # inpaint_simulation, ground_truth, simulated_part 
    return inpainted_final, x_real_raw, simulated_part, cond_raw

In [ ]:
inpaint_simulation, ground_truth, simulated_part, x_cond = forward_simulate(steps=sim_steps, batch=batch, diffusion=diffusion)

In [ ]:
simulated_part.shape

In [ ]:
ground_truth.shape

In [ ]:
x_cond.shape

In [ ]:
inspect_data(inpaint_simulation,name='inpaint simulation')
inspect_data(ground_truth,name='ground truth')
inspect_data(ground_truth[:, -sim_steps:, :, :],name='ground truth latest n steps part')
inspect_data(simulated_part,name='simulated part')

In [ ]:
unscaled_inpaint_simulation = inverse_scale_with_cond(inpaint_simulation.cpu().numpy(), x_cond.cpu().numpy() , scaler)
unscaled_ground_truth = inverse_scale_with_cond(ground_truth.cpu().numpy(), x_cond.cpu().numpy() , scaler)
unscaled_simulated_part = inverse_scale_with_cond(simulated_part, x_cond[:, -sim_steps:, :, :].cpu().numpy() , scaler)

inspect_data(unscaled_inpaint_simulation,name='inpaint simulation')
inspect_data(unscaled_ground_truth,name='ground truth')
inspect_data(unscaled_ground_truth[:,-sim_steps:, :, :],name='ground truth at simulate part')
inspect_data(unscaled_simulated_part,name='simulated part')

In [ ]:
def gbm_monte_carlo_simulate(past_log_returns, n_steps, n_samples=100):
    """
    Simulate future prices using Geometric Brownian Motion (GBM)
    Args:
        past_log_returns: [Time, Assets] (numpy array)
        n_steps: steps to sim
        n_samples: n paths to simulate
    """
    # 1. คำนวณ Stats จากข้อมูลในอดีต (batch นี้)
    # mu = drift (แนวโน้ม), sigma = volatility (ความผันผวน)
    mu = np.mean(past_log_returns, axis=0)
    sigma = np.std(past_log_returns, axis=0)
    
    n_assets = past_log_returns.shape[1]
    
    # 2. จำลองอนาคต (Simulate Future Log Returns)
    # ภายใต้สมมติฐาน GBM: Log Return จะเป็น Normal Distribution
    # ret ~ N(mu, sigma)
    
    # สร้าง array เปล่า [Samples, Steps, Assets]
    sim_log_returns = np.random.normal(
        loc=mu, 
        scale=sigma, 
        size=(n_samples, n_steps, n_assets)
    )
            
    return sim_log_returns

In [ ]:
mc_simulations = gbm_monte_carlo_simulate(unscaled_ground_truth[:, :-sim_steps, :, :].squeeze(-1).squeeze(0), sim_steps, num_sims)
# n_sample, n_steps, n_assets
mc_simulations.shape

In [ ]:
type(unscaled_simulated_part)

In [ ]:
# sim_part_df = pd.DataFrame(unscaled_simulated_part[0].squeeze(-1))
# Case Batch size = 1
sim_part_df = pd.DataFrame(unscaled_simulated_part.squeeze(-1).squeeze(0))
sim_part_df

In [ ]:
mc_simulations_df = pd.DataFrame(mc_simulations[0]) 
mc_simulations_df

In [ ]:
# gt_sim_part_df = pd.DataFrame(unscaled_ground_truth[0, -10:, :, :].squeeze(-1))

# Case Batch size = 1
gt_sim_part_df = pd.DataFrame(unscaled_ground_truth[:, -sim_steps:, :, :].squeeze(-1).squeeze(0))
gt_sim_part_df

In [ ]:
plotting.plot_covariance(risk_models.sample_cov(gt_sim_part_df), plot_correlation=True)
plotting.plot_covariance(risk_models.sample_cov(sim_part_df), plot_correlation=True)
plotting.plot_covariance(risk_models.sample_cov(mc_simulations_df), plot_correlation=True)

In [ ]:
batch_expanded = {
    "x": batch["x"].repeat(num_sims, 1, 1, 1),           # [num_sims, 64, 14, 1]
    "x_cond": batch["x_cond"].repeat(num_sims, 1, 1, 1)  # [num_sims, 64, 14, 6]
}


# inpaint_simulation, ground_truth, simulated_part, x_cond
inpaint_final, _, simulated_part, x_cond = forward_simulate(
    steps=sim_steps,
    batch=batch_expanded,
    diffusion=diffusion
)
genai_simulations = simulated_part
genai_simulations.shape

In [ ]:
genai_simulations.shape

In [ ]:
unscaled_genai_simulated = inverse_scale_with_cond(genai_simulations[:, :, : ,:], batch["x_cond"].repeat(num_sims, 1, 1, 1)[:, -sim_steps:, :, :].cpu().numpy() , scaler)
inspect_data(unscaled_genai_simulated,name='GenAI simulations')

In [ ]:
unscaled_genai_simulated[0,:, :, :].squeeze(-1).cumsum(0).shape

In [ ]:
genai_simulations = unscaled_genai_simulated[0].squeeze(-1)
n_steps, n_assets = genai_simulations.shape

for i in range(n_assets):
    plt.plot(genai_simulations[:, i].cumsum(0), label=f"{i}")
plt.legend()
plt.show()

In [ ]:
mc_simulations.shape

In [ ]:
n_sims, n_steps, n_assets = mc_simulations.shape

for i in range(n_assets):
    plt.plot(mc_simulations[0, :, i].cumsum(0), label=f"{i}")
plt.legend()
plt.show()

In [ ]:
inspect_data(unscaled_ground_truth[:,:-10, :, :], name="Ground Truth")

In [ ]:
ground_truth = unscaled_ground_truth[:, -steps_sim:, :, :].squeeze(-1).squeeze(0)
n_obs, n_assets = ground_truth.shape

for i in range(n_assets):
    plt.plot(ground_truth[:, i].cumsum(0), label=f"{i}")
plt.legend()
plt.show()